# Qwen3.5-9B (4-bit) on Google Colab: Experiment 1 inference + Experiment 2 analysis (single self-contained notebook)

Runs **Qwen3.5-9B in 4-bit** (`RedHatAI/Qwen3.5-9B-quantized.w4a16`: GPTQ INT4 weights, BF16 activations; Apache-2.0, not gated) with **vLLM 0.32.0**, thinking **on**, then
1. runs the E1 prompts **B and C** on the 2,670 Who&When Pro multi-agent text traces (same prompts, parser and scorer as the luna and MiMo runs),
2. scores E1 (luna + MiMo + Qwen) and re-runs the E2 analysis with Qwen included.

**Nothing to upload.** The code, prompts and the luna and MiMo result logs are embedded in this notebook; the dataset (`text.jsonl`, `taxonomy.yaml`) is downloaded from Hugging Face `Leoxx/whowhen_pro` and checked against the SHA-256 of the copy used for the earlier runs; the official renderers are cloned from GitHub at a pinned commit.

**Before you start**
- **GPU:** a bigger GPU is recommended for speed (A100 40/80 GB or H100). The model needs only about 11 GB of weights; the notebook scales concurrency to the GPU (24 / 64 / 128 parallel requests for 24 GB / 40 GB / 80 GB+). In VS Code: Select Kernel > Colab > New Colab Server > pick the GPU. Without a GPU the first cell stops with a clear message.
- **Settings used (all in the first cells and in `config/models.yaml`):** thinking on; sampling from the Qwen3.5 model card for thinking mode on general tasks (temperature 1.0, top_p 0.95, top_k 20, min_p 0, presence_penalty 1.5, seed 0); max output 32,768 tokens including thinking (the same cap as luna); context 57,344.
- **4-bit disclosure:** only the weights of the linear layers in the transformer blocks are INT4 (GPTQ, group size 128); the linear-attention layers, embeddings and vision encoder stay in BF16 (checkpoint is 11 GB, not 5 GB). Results are for this quantized model, not the BF16 original; the manifest records it.
- **Time and sessions:** outputs are appended to disk and are **resumable**: after a disconnect, re-run the cells up to the server start and then the full-run cell; finished calls are skipped. Without Drive, results live only on the runtime until the last cell zips them, so **download the results zip regularly**.
- Versions were verified to **resolve together** with `uv` for Linux x86_64 / Python 3.12, and the server cell was written from real Colab failures seen with another model (missing `nvcc`/`ninja`, an API-key variable making the server reject requests). It has **not yet been run with Qwen on a GPU**: if a cell fails, the server cell prints the real error lines.

## 1. GPU check and settings

In [ ]:
import subprocess, json, os, re, time, sys
try:
    q = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version,compute_cap", "--format=csv,noheader,nounits"], capture_output=True, text=True)
except FileNotFoundError:
    raise SystemExit("This Colab runtime has NO GPU (nvidia-smi not found). Disconnect and reconnect with a GPU machine: "
                     "in the Colab web page use Runtime > Change runtime type > pick a GPU; in VS Code use Select Kernel > Colab > New Colab Server and pick a GPU machine type "
                     "(an A100 or H100 is recommended for speed; a T4 16 GB is too small for a comfortable run).")
print(q.stdout or q.stderr)
assert q.returncode == 0 and q.stdout.strip(), "nvidia-smi failed; no usable GPU on this runtime."
GPU_NAME, MEM_MB, DRIVER, CC = [x.strip() for x in q.stdout.strip().splitlines()[0].split(",")]
MEM_GB = float(MEM_MB) / 1024
print(f"GPU {GPU_NAME} | {MEM_GB:.0f} GB | driver {DRIVER} | compute capability {CC} | python {sys.version.split()[0]}")

# 4-bit checkpoint (vLLM reads the quantization from its config). Alternatives, if this one fails to load:
#   "QuantTrio/Qwen3.5-9B-AWQ"  (AWQ 4-bit)   or   "cyankiwi/Qwen3.5-9B-AWQ-4bit"  (AWQ via compressed-tensors)
MODEL = "RedHatAI/Qwen3.5-9B-quantized.w4a16"
THINKING = True                          # E1 setting for Qwen: thinking ON; one setting for prompts B and C
MAX_MODEL_LEN = 57344                    # longest E1 prompt is 23,927 tokens + 32,768 max output tokens
# concurrency: weights are about 11 GB and the hybrid architecture (24 of 32 layers are linear attention) has a very small KV cache
if MEM_GB >= 75:   PRESET = dict(util=0.90, seqs=128)
elif MEM_GB >= 38: PRESET = dict(util=0.90, seqs=64)
elif MEM_GB >= 22: PRESET = dict(util=0.90, seqs=24)
else: raise SystemExit(f"{MEM_GB:.0f} GB is too small; use a GPU with at least 24 GB (an A100 or H100 is better).")
print("PRESET:", PRESET, "| model:", MODEL, "| thinking:", THINKING)

## 2. Storage location (Drive if available, otherwise local `/content`)

In [ ]:
import os, sys, json, time, zipfile, subprocess, base64, lzma, tarfile, io, hashlib, shutil, re   # imported here too, so each cell also works on its own
USING_DRIVE = False
try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT, USING_DRIVE = "/content/drive/MyDrive/whowhen_colab", True
except Exception as e:
    print(f"Drive mount not available here ({type(e).__name__}: {str(e)[:150]}). Using local /content (download the results at the end).")
    PROJECT = "/content/whowhen_colab"
os.makedirs(PROJECT, exist_ok=True)
E1, E2 = f"{PROJECT}/experiment_1", f"{PROJECT}/experiment_2"
print("USING_DRIVE =", USING_DRIVE, "| PROJECT =", PROJECT)

## 3. Embedded project files (code, prompts, configs: 21 files, xz + base64). Do not edit; run to write them to disk.

In [ ]:
CODE_B64 = """
...
"""

def unpack(b64, dest):
    data = lzma.decompress(base64.b64decode("".join(b64.split())))
    with tarfile.open(fileobj=io.BytesIO(data)) as t:
        t.extractall(dest)
        return t.getnames()
names = unpack(CODE_B64, PROJECT)
print(len(names), "files written under", PROJECT); print(*names, sep="\n")

## 4. Embedded E1 results for luna and MiMo (needed only for scoring and E2; MiMo is **partial**: B 1,894 and C 1,196 of 2,670 traces)

In [ ]:
LOGS_B64 = """
...
"""
names = unpack(LOGS_B64, PROJECT)
for f in names:
    print(f, sum(1 for _ in open(f"{PROJECT}/{f}")), "rows")

## 5. Dataset from Hugging Face (public), with integrity checks

In [ ]:
DATA_BASE = "https://huggingface.co/datasets/Leoxx/whowhen_pro/resolve/main"
os.makedirs(f"{PROJECT}/data", exist_ok=True)
EXPECT = {"data/text.jsonl": "51403bfd1b6570d0a9f8e62d25a50aa62504cb1264c33a3627ead893b7e2dd51", "taxonomy.yaml": "f5d77edee84622e025519c31404a5b741ff4e008171392e35df0cceebd41d140"}   # SHA-256 of the copies used for the luna and MiMo runs
for remote, local in (("data/text.jsonl", "data/text.jsonl"), ("taxonomy.yaml", "data/taxonomy.yaml")):
    dest = f"{PROJECT}/{local}"
    if not os.path.exists(dest):
        subprocess.run(["curl", "-sL", "--fail", "-o", dest, f"{DATA_BASE}/{remote}"], check=True)
    got = hashlib.sha256(open(dest, "rb").read()).hexdigest()
    assert got == EXPECT[remote], f"{remote}: SHA-256 mismatch ({got}); the dataset on the Hub changed. Stop and tell the author."
    print("OK", local, got[:16])
n = sum(1 for _ in open(f"{PROJECT}/data/text.jsonl")); assert n == 6257, n
print("text.jsonl rows:", n)
shutil.copy(f"{PROJECT}/data/taxonomy.yaml", f"{E1}/prompts/taxonomy.yaml")      # the 17-mode taxonomy used by prompt C

## 6. Install (isolated venv, so Colab's own packages are not touched)
Pins are in `colab/requirements-colab.txt`. Key constraints: **vLLM 0.32.0 needs `transformers>=5.16.1,<5.20.0`** (the newest transformers does not work), `numpy<2.4`, `huggingface_hub 1.x`. `uv --torch-backend=auto` picks the torch/CUDA wheel that matches the Colab driver. About 5-10 minutes.

In [ ]:
!pip -q install uv
!uv venv /content/venv --python 3.12
!uv pip install --python /content/venv/bin/python --torch-backend=auto -r {PROJECT}/colab/requirements-colab.txt
!/content/venv/bin/python -c "import torch, vllm, transformers, huggingface_hub, openai, numpy, pandas; print('torch', torch.__version__, '| cuda', torch.version.cuda, '| gpu', torch.cuda.is_available()); print('vllm', vllm.__version__, '| transformers', transformers.__version__, '| huggingface_hub', huggingface_hub.__version__, '| openai', openai.__version__, '| numpy', numpy.__version__, '| pandas', pandas.__version__)"

## 7. Renderer repo (pinned), model name in the config, and unit tests

In [ ]:
!rm -rf /content/whowhen_pro && git clone -q https://github.com/ag2ai/whowhen_pro /content/whowhen_pro && git -C /content/whowhen_pro checkout -q 14369dcb079c7cdf86ed2924145d5a28ff5c73b2
os.environ["WHOWHEN_EVAL_REPO"] = "/content/whowhen_pro"      # official per-framework renderers (commit 14369dc) used by experiment_1/src/data.py
os.environ["VLLM_API_KEY"] = "EMPTY"                          # the runner's client sends this; the SERVER is started without it (see the server cell)
# make the E1 config match the model chosen in the first cell (the runner reads model_id and max_model_len from config/models.yaml)
cfg = f"{E1}/config/models.yaml"; s = open(cfg).read(); head, qwen = s.split("  qwen:")
qwen = re.sub(r'model_id:\s*"[^"]+"', f'model_id: "{MODEL}"', qwen); qwen = re.sub(r"max_model_len:\s*\d+", f"max_model_len: {MAX_MODEL_LEN}", qwen)
open(cfg, "w").write(head + "  qwen:" + qwen)
print([l.strip() for l in qwen.splitlines() if "model_id" in l or "max_model_len" in l or "max_output_tokens" in l or "sampling" in l])
!cd {E1} && /content/venv/bin/python -m pytest -q tests

## 8. Start the vLLM server (cleans up, tries safer settings in turn, accepts only a server that really generates)
Flags: `--reasoning-parser qwen3` keeps the model's thinking out of the scored text; `--default-chat-template-kwargs {"enable_thinking": true}` pins thinking on (Qwen3.5's default) without adding a system message; `--language-model-only` skips the vision encoder (text only, more KV-cache memory). Lessons built in from another model on Colab: there is no `nvcc`/`ninja` (so `ninja` is installed and the FlashInfer sampler is switched off), and `VLLM_API_KEY` is **not** passed to the server (vLLM would treat it as a required key and answer 401). Hybrid (linear-attention) models can hit a CUDA-graph cache limit, which attempt 2 handles.

In [ ]:
import subprocess, urllib.request, signal, re, time, os, sys, json, shutil

def gpu_used_mb():
    try:
        r = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"], capture_output=True, text=True)
        return int(r.stdout.strip().splitlines()[0])
    except Exception:
        return 0

def kill_vllm():
    """Kill every leftover vLLM process (a failed start can leave the engine holding GPU memory) and wait for the memory to be freed."""
    me = os.getpid()
    for l in subprocess.run(["ps", "-eo", "pid,args"], capture_output=True, text=True).stdout.splitlines()[1:]:
        pid, _, args = l.strip().partition(" ")
        if pid.isdigit() and int(pid) != me and re.search(r"vllm serve|EngineCore|VLLM::|vllm\.v1|vllm\.entrypoints", args):
            try: os.kill(int(pid), signal.SIGKILL)
            except ProcessLookupError: pass
    for _ in range(40):
        if gpu_used_mb() < 1500: return True
        time.sleep(2)
    return False

VENV = globals().get("VENV", "/content/venv/bin")
LOGDIR = globals().get("LOGDIR", "/content")
# Colab has no nvcc/ninja, so FlashInfer cannot JIT-compile kernels. Install ninja into the venv and switch the FlashInfer sampler off.
uv = shutil.which("uv")
if uv and not os.path.exists(f"{VENV}/ninja"):
    subprocess.run([uv, "pip", "install", "--python", f"{VENV}/python", "ninja"], capture_output=True, text=True)
# NOTE: an earlier cell sets VLLM_API_KEY="EMPTY" for the runner; vLLM treats that variable as a REQUIRED api key (-> HTTP 401), so it is left out of the server env.
ENV = {"VLLM_USE_FLASHINFER_SAMPLER": "0", "PATH": f"{VENV}:{os.environ.get('PATH', '')}"}

base = ["--port", "8000", "--max-model-len", str(MAX_MODEL_LEN), "--max-num-seqs", str(PRESET["seqs"]),
        "--reasoning-parser", "qwen3", "--default-chat-template-kwargs", json.dumps({"enable_thinking": THINKING})]
LMO = ["--language-model-only"]
UTIL = ["--gpu-memory-utilization", str(PRESET["util"])]
EAGER = ["--enforce-eager", "--gpu-memory-utilization", "0.85"]
# progressively safer settings; an attempt only counts if the server starts AND really generates a completion
ATTEMPTS = [("default (CUDA graphs), text-only", base + LMO + UTIL),
            ("CUDA graphs capped to the batch size (hybrid-model cache limit)", base + LMO + UTIL + ["--max-cudagraph-capture-size", str(PRESET["seqs"])]),
            ("eager mode (no torch.compile / CUDA graphs), text-only", base + LMO + EAGER),
            ("eager mode, multimodal loader, no prefix caching", base + EAGER + ["--no-enable-prefix-caching"])]

def root_cause(path, first=10, last=6):
    lines = open(path, errors="replace").read().splitlines()
    pat = re.compile(r"(Error|ERROR|error:|out of memory|OutOfMemory|CUDA|nvcc|ninja|not supported|Unsupported|Failed|insufficient|No such file|Cannot|cannot|unrecognized arguments)")
    skip = ("File ", "^", "raise ", "return ", "async ", "with ")
    hits = [(i, l) for i, l in enumerate(lines) if pat.search(l) and not l.lstrip().startswith(skip) and "Engine core initialization failed" not in l
            and "See root cause above" not in l and "Capturing CUDA graphs" not in l and "FlashInfer kernels are disabled" not in l]
    print(f"   {len(lines)} log lines; error-like lines (first {first}, last {last}):")
    for i, l in hits[:first]: print(f"   [{i}] {l[:260]}")
    if len(hits) > first + last: print("   ...")
    for i, l in hits[max(first, len(hits) - last):]: print(f"   [{i}] {l[:260]}")
    frames = [l.strip() for l in lines if re.search(r'File ".*/(vllm|flashinfer|torch|triton)/', l)]
    print("   deepest vllm/flashinfer/torch frames:"); [print("    ", f[:230]) for f in frames[-6:]]

def wait_ready(p, timeout=3600):
    t0 = time.time()
    while time.time() - t0 < timeout:
        if p.poll() is not None: return False
        try:
            if urllib.request.urlopen("http://localhost:8000/health", timeout=5).status == 200: return True
        except Exception: pass
        time.sleep(10)
    return False

def generation_ok():
    """Send a real request. Returns (ok, detail)."""
    body = json.dumps({"model": MODEL, "messages": [{"role": "user", "content": "What is 17*23? Answer in one short sentence."}],
                       "max_tokens": 4000, "temperature": 1.0, "top_p": 0.95, "top_k": 20, "presence_penalty": 1.5}).encode()
    req = urllib.request.Request("http://localhost:8000/v1/chat/completions", data=body, headers={"Content-Type": "application/json"})
    try:
        r = json.loads(urllib.request.urlopen(req, timeout=900).read())
    except urllib.error.HTTPError as e:
        return False, f"HTTP {e.code}: {e.read().decode(errors='replace')[:800]}"
    except Exception as e:
        return False, f"{type(e).__name__}: {e}"
    return ("choices" in r), (json.dumps(r)[:800] if "choices" not in r else "")

ok, proc, SERVE_CMD = False, None, None
for k, (name, args) in enumerate(ATTEMPTS, 1):
    freed = kill_vllm()
    print(f"\n=== attempt {k}/{len(ATTEMPTS)}: {name} | GPU memory in use before start: {gpu_used_mb()} MB (cleanup ok: {freed})")
    SERVE_CMD = [f"{VENV}/vllm", "serve", MODEL] + args
    print(" ".join(SERVE_CMD))
    log = f"{LOGDIR}/vllm_attempt{k}.log"
    proc = subprocess.Popen(SERVE_CMD, stdout=open(log, "w"), stderr=subprocess.STDOUT, env={**{k_: v for k_, v in os.environ.items() if k_ != "VLLM_API_KEY"}, **ENV}, start_new_session=True)
    ok = wait_ready(proc)
    if ok:
        ok, detail = generation_ok()
        if ok:
            shutil.copy(log, f"{LOGDIR}/vllm_server.log"); print(f"SERVER READY AND GENERATING (attempt {k}: {name})"); break
        print(f"attempt {k}: server is up but the test request failed: {detail}")
    else:
        print(f"attempt {k}: server did not start.")
    print(f"Root cause from {log}:"); root_cause(log)
    try: os.killpg(proc.pid, signal.SIGKILL)
    except Exception: pass
if not ok: print("\nALL ATTEMPTS FAILED. Send the text printed above (and `!nvidia-smi`) for diagnosis.")
assert ok, "vLLM server did not start and generate; see the root causes printed above"

### Sanity request (thinking on, final text separate from the thoughts)

In [ ]:
import requests
resp = requests.post("http://localhost:8000/v1/chat/completions", json={"model": MODEL, "messages": [{"role": "user", "content": "What is 17*23? Answer in one short sentence."}],
                     "max_tokens": 4000, "temperature": 1.0, "top_p": 0.95, "top_k": 20, "presence_penalty": 1.5}, timeout=900)
r = resp.json()
if "choices" not in r:                      # show the server's error instead of a bare KeyError
    print("HTTP", resp.status_code, "\n", json.dumps(r)[:1500], "\n--- server log tail ---\n", open("/content/vllm_server.log").read()[-3000:])
    raise SystemExit("The server answered with an error, see above.")
msg = r["choices"][0]["message"]
print("content   :", msg.get("content"))
print("reasoning :", ((msg.get("reasoning_content") or msg.get("reasoning") or "")[:300]) or "(none)")
print("usage     :", r["usage"])
assert msg.get("content") and "<think>" not in msg["content"] and "</think>" not in msg["content"], "thoughts leaked into the scored text; check --reasoning-parser qwen3"
if THINKING: assert (msg.get("reasoning_content") or msg.get("reasoning")), "thinking is ON but no reasoning came back"

### Record the environment in the manifest (before any E1 call)

In [ ]:
import datetime, hashlib
freeze = subprocess.run(["uv", "pip", "freeze", "--python", f"{VENV}/python"], capture_output=True, text=True).stdout
open(f"{E1}/E1_qwen_colab_pip_freeze.txt", "w").write(freeze)
sha = lambda p: hashlib.sha256(open(p, "rb").read()).hexdigest()
man = f"""
## Qwen3.5-9B 4-bit on Colab ({datetime.datetime.utcnow().isoformat(timespec='seconds')}Z)
- Model: {MODEL} (Apache-2.0), served by vLLM; quantization: 4-bit GPTQ INT4 weights (group size 128) for the linear layers of the transformer blocks, activations BF16; linear-attention layers, embeddings and vision encoder stay BF16. NOT the BF16 original.
- GPU: {GPU_NAME}, {MEM_GB:.0f} GB, driver {DRIVER}, python {sys.version.split()[0]}; pip freeze in E1_qwen_colab_pip_freeze.txt
- Server command: `{' '.join(SERVE_CMD)}`
- Thinking: {"ON via --default-chat-template-kwargs enable_thinking=true; --reasoning-parser qwen3 (thoughts excluded from the scored text)" if THINKING else "OFF"}; one setting for prompts B and C
- Sampling (config/models.yaml): temperature 1.0, top_p 0.95, top_k 20, min_p 0, presence_penalty 1.5, seed 0 (Qwen3.5 model card, thinking mode, general tasks); max output tokens 32768 incl. thinking; max_model_len {MAX_MODEL_LEN}; single user message, no system prompt; concurrency {PRESET["seqs"]} sequences
- Hashes: prompts/B.txt {sha(E1+"/prompts/B.txt")[:16]}, prompts/C.txt {sha(E1+"/prompts/C.txt")[:16]}, src/parser.py {sha(E1+"/src/parser.py")[:16]}, src/score.py {sha(E1+"/src/score.py")[:16]}, src/run_local.py {sha(E1+"/src/run_local.py")[:16]}
- vLLM is not fully deterministic. Deviations: 4-bit quantized model; sampling per the model card (not temperature 0); presence_penalty 1.5 is part of the recommended thinking-mode sampling.
"""
open(f"{E1}/E1_manifest.md", "a").write(man); print(man)

## 9. Smoke test (12 traces x prompts B and C): parseability, truncation, latency only. Correctness is NOT looked at.
Pass rule from the plan: at most 1 parse failure in 12 per prompt (truncation counts as a failure). If it fails, stop and ask; do not edit prompts, parser or settings.

In [ ]:
!cd {E1} && /content/venv/bin/python src/run_local.py --stage smoke --models qwen --prompts B C --workers {max(1, PRESET["seqs"] // 2)} 2>&1 | tail -30

In [ ]:
import sys; sys.path.insert(0, f"{E1}/src")
import glob, statistics as st, pandas as pd
from parser import parse_response
def summarize(model="qwen", stage="smoke"):
    """Latest SUCCESSFUL response per trajectory (same rule as the runner and the scoring). Old error rows are counted separately."""
    rows, errs = [], {}
    for f in sorted(glob.glob(f"{E1}/outputs/{stage}/{model}/*.jsonl")):
        last = {}
        for l in open(f):
            d = json.loads(l)
            if d.get("error"):
                errs[d["prompt_id"]] = errs.get(d["prompt_id"], 0) + 1; continue
            last[d["trajectory_id"]] = d
        for d in last.values():
            pr = parse_response(d.get("text"), d.get("finish_reason"))
            rows.append({"prompt": d["prompt_id"], "ok": pr["ok"], "failure": pr["failure"], "finish": d["finish_reason"], "in": d.get("input_tokens"), "out": d.get("output_tokens"), "lat": d["latency_s"]})
    return pd.DataFrame(rows), errs
S, ERRS = summarize()
print("old error rows (NOT counted):", ERRS)
print(S.groupby("prompt").agg(n=("ok", "size"), parse_failures=("ok", lambda s: int((~s).sum())), truncated=("finish", lambda s: int((s == "length").sum())),
                              out_mean=("out", "mean"), out_max=("out", "max"), lat_mean=("lat", "mean"), lat_max=("lat", "max")).round(1))
print("failure reasons:", S[~S.ok].groupby(["prompt", "failure"]).size().to_dict())
bad = S.groupby("prompt").ok.apply(lambda s: int((~s).sum()))
print("SMOKE", "PASS" if bad.max() <= 1 else "FAIL: stop and review", "| parse failures per prompt (truncation counts as a failure):", bad.to_dict(), "| rule: at most 1 in 12")
calls, conc = 2 * 2670, PRESET["seqs"]
print(f"Rough projection for 5,340 calls at concurrency {conc} (latency-based; smoke is a small batch, the real rate is usually better): ~{S.lat.mean() * calls / conc / 3600:.1f} h")

## 10. Full run (all 2,670 traces, prompts B and C): resumable, appends to disk
One process per prompt, each with half the server's slots, so requests never queue past the client timeout. Stops itself if more than 25% of calls fail after the first 40. Re-running this cell after a disconnect skips finished calls (failed or API-error rows are retried).

In [ ]:
procs = []
for p in ["B", "C"]:
    log = open(f"{E1}/outputs/full_qwen_{p}.log", "a")
    procs.append(subprocess.Popen([f"{VENV}/python", "src/run_local.py", "--stage", "full", "--models", "qwen", "--prompts", p, "--workers", str(max(1, PRESET["seqs"] // 2))],
                                  cwd=E1, stdout=log, stderr=subprocess.STDOUT, env=os.environ.copy()))
print("started", [p.pid for p in procs])

In [ ]:
# Monitor (re-run any time). Shows progress, error rows and parse failures.
import glob, pandas as pd
from parser import parse_response
def progress():
    out = {}
    for p in ["B", "C"]:
        f = f"{E1}/outputs/raw/qwen/{p}.jsonl"
        rows = [json.loads(l) for l in open(f)] if os.path.exists(f) else []
        ok = [r for r in rows if not r.get("error")]
        out[p] = {"done": len({r["trajectory_id"] for r in ok}), "of": 2670, "error_rows": len(rows) - len(ok),
                  "parse_fail": sum(1 for r in ok if not parse_response(r["text"], r["finish_reason"])["ok"]),
                  "truncated": sum(1 for r in ok if r["finish_reason"] == "length"), "out_tokens": sum(r.get("output_tokens") or 0 for r in ok)}
    return out
P1 = progress(); t1 = time.time(); print(pd.DataFrame(P1).T)
print("processes alive:", [p.poll() is None for p in procs] if "procs" in globals() else None, "| server alive:", proc.poll() is None)

In [ ]:
time.sleep(120); P2 = progress(); dt = time.time() - t1
for p in ["B", "C"]:
    rate = (P2[p]["done"] - P1[p]["done"]) / dt * 3600
    left = 2670 - P2[p]["done"]
    print(f"prompt {p}: {P2[p]['done']}/2670 | {rate:.0f} calls/h | {P2[p]['out_tokens'] - P1[p]['out_tokens']} new output tokens in {dt:.0f}s | ETA {left / rate:.1f} h" if rate > 0 else f"prompt {p}: {P2[p]['done']}/2670 (no progress in {dt:.0f}s)")

## 11. E1 scoring (luna + MiMo from the embedded logs, Qwen from this run) and token/latency analysis
Run after both Qwen prompts reach 2,670 (partial results are scored too, but the summary only includes complete models).

In [ ]:
!cd {E1} && /content/venv/bin/python src/analyze.py 2>&1 | tail -40
!cd {E1} && /content/venv/bin/python src/tokens_latency.py 2>&1 | head -30

In [ ]:
display(pd.read_csv(f"{E1}/E1_summary.csv"))
display(pd.read_csv(f"{E1}/outputs/E1_paired.csv"))

## 12. Experiment 2 analysis with Qwen included
Primary GEEs use mathchat (reference) vs magentic-one, prompts B (reference) and C, clusters = task seed group. Pre-specified: if a primary fit fails to converge or warns, the script stops and writes `E2_GEE_errors.txt`; do **not** change the model, ask first. E2 was planned for complete E1 results; MiMo is still partial and this is stated in the notes.

In [ ]:
!cd {E2} && /content/venv/bin/python src/build.py && /content/venv/bin/python src/primary.py 2>&1 | head -40

In [ ]:
!cd {E2} && /content/venv/bin/python src/run_all.py 2>&1 | tail -60
!cd {E2} && /content/venv/bin/python src/breakdown.py && /content/venv/bin/python src/figures.py

In [ ]:
display(pd.read_csv(f"{E2}/E2_effects_table.csv").query("Kind == 'primary'")[["Outcome", "Effect", "Odds ratio", "95% CI", "p-value (raw)", "p-value (Holm, primary only)"]])
print(open(f"{E2}/E2_GEE_errors.txt").read())

## 13. Package the results and shut the server down
With Drive, everything is already persisted in `PROJECT`. Without Drive, download `/content/whowhen_qwen_results.zip` yourself (Colab file panel / VS Code Colab server explorer); `files.download` only works in the Colab web page.

In [ ]:
import shutil
out = "/content/whowhen_qwen_results"
shutil.copy("/content/vllm_server.log", f"{E1}/outputs/vllm_server.log")
try: os.killpg(proc.pid, signal.SIGKILL)       # stop the server
except Exception: pass
shutil.make_archive(out, "zip", PROJECT)       # whole project folder (outputs, tables, figures, logs)
print("results zip:", out + ".zip", f"({os.path.getsize(out + '.zip') / 1e6:.1f} MB)", "| persisted on Drive" if USING_DRIVE else "| NOT on Drive: download it now")
try:
    from google.colab import files
    files.download(out + ".zip")
except Exception as e:
    print("files.download not available here:", type(e).__name__, "- fetch the zip from the file panel.")